# Forzy EDA corrected for sensor semantics

This notebook orchestrates the versioned `twinops.ml` package; it contains no exclusive feature or scoring logic. `velocity_rms` is vibration velocity RMS in mm/s, not shaft RPM. Acceleration is retained for audit and excluded from the official score while its statistic is unknown. S1-S2 physical comparison is excluded until mounting and axis are confirmed.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from twinops.ml.backtest import run_backtest_csv


In [ ]:
DATA_PATH = Path(os.environ.get('FORZY_CURATED_CSV', 'data/forzy-curated-features.csv'))
OUTPUT_PATH = Path('../artifacts/ml')
if DATA_PATH.name == 'readings_rows.csv':
    raise ValueError('readings_rows.csv is a 677-row mock with another schema and is forbidden here')
dataset_status = {
    'expectedRows': 7183,
    'path': str(DATA_PATH),
    'available': DATA_PATH.is_file(),
}
dataset_status


In [ ]:
if DATA_PATH.is_file():
    curated = pd.read_csv(DATA_PATH)
    dataset_status['actualRows'] = len(curated)
    report = run_backtest_csv(DATA_PATH, OUTPUT_PATH).to_dict()
else:
    curated = None
    report = {
        'status': 'not_executed_missing_real_dataset',
        'cycle_results': [],
        'candidate_events': [],
    }
report


In [ ]:
candidate_times = {'13:48:10', '13:56:10'}
review = [
    event for event in report.get('candidate_events', [])
    if pd.Timestamp(event['observed_at']).strftime('%H:%M:%S') in candidate_times
]
{
    'candidateReview': review,
    'interpretation': 'candidate_not_ground_truth',
    'failureClassification': 'not_performed',
}
